# Lab 4 — The Machine Picks Its Own Features

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/INCORTX/INCORTX.github.io/blob/master/DataAnalytics/session-04/lab_04.ipynb)

**Data Analytics · Lab session 4**

Three sessions of telling the model which columns to look at. **This session the model decides for
itself** — a neural network from Lecture 6 (ANN and CNN) and Lecture 7 (CNN), written in PyTorch, first
on a table you already know and then on pictures, where there are no columns at all.

**The demo is one story in two acts.** Act one follows the slides: a neuron, the four-line training
loop, and a network on the titanic table from session 1 — which **loses to session 1's tree**, 0.7877
against 0.8101. Act two looks at what the loss number hid: a validation curve that says the network had
memorised the rows by epoch 22; a dense layer on 10,000 pictures that scores 0.8349 and then **falls
to 0.5392 when every picture is moved two pixels**; and a convolutional network that keeps 0.7395 on
the same moved pictures with eleven times fewer weights — because it learned *what* the shapes are,
not *where* they sat.

**No GPU is needed for anything in this file.** Every training cell finishes in under fifteen seconds on
a plain Colab CPU.

### How the session runs

| | Part | What happens |
|---|---|---|
| **1** | 🎤 **Assignment 3 on screen — 60 min** | **The hour opens with presentations of last session's work.** Two speakers per group, 5+ minutes, then two of questions. |
| **2** | 🎬 **Demo — 45 min** | Blocks A and B. The instructor walks them; you watch. Do not type along — you keep this file. |
| **3** | 📋 **Pick a topic** | Your group claims one of the twenty. First come, first served. |
| **4** | 🟠 **Your hour — 60 min** | The section at the bottom. The same five steps as before — with a network in step 4. |

---
## 🎤 The 45 minutes we actually walk through

**The demo half of this notebook holds more than the slot.** The seven below are the ones we walk
together. Everything else is reference you keep. *(This table is a map, not something read aloud.)*

| | Walked in the demo | Why this one earns the time |
|:--:|---|---|
| 1 | **How the session runs** | so the hour is not spent guessing what to hand in |
| 2 | **A.1** — a neuron on a table you know | the network from Lecture 6, the four-line loop, and **a loss that fell while the accuracy lost** |
| 3 | **A.2** — the curve that explains the number | train loss against validation loss 🆕 — the network memorised the rows at epoch 22 |
| 4 | **B.1** — convolution on the slide's own numbers | Lecture 7's 585, then the same operation on a real picture |
| 5 | **B.2** — flatten it, then move it two pixels | a dense layer loses 30 points to a two-pixel shift; a CNN loses 9 |
| 6 | **B.3** — what the network found | the eight kernels nobody wrote |
| 7 | **Part 2 — picking your topic** | you skim and claim, not read out loud |

**The two 📖 cells are yours to read** — about 4 minutes if you sit down with them: six curve shapes and
what to do about each, and this session's chart-polish rung.

> **Everything in this file is classification** — a label per row, accuracy and F1 from session 1. What
> is new is the model, and the curve you read before you trust its number.

---
---
# 🎬 Part 1 — The Demo · blocks A and B

**Watch, do not type along.** The header of each subsection says whether it is walked live (🎤)
or reference (📖).

---
# A · A Neuron on a Table You Already Know
🎤 **Walked live: all of A.** The network from Lecture 6, on the titanic rows from session 1 — and the
number it loses to.

In [ ]:
import warnings; warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
pd.set_option('display.width', 120)
plt.rcParams['figure.figsize'] = (7, 4)
plt.rcParams['axes.grid'] = True; plt.rcParams['grid.alpha'] = 0.3
print('torch', torch.__version__, '- ready')

**Three helpers for this file: `fit()`, `accuracy()` and `plot_curve()`.** Run the cell once. Do not read
it now — A.1 writes the loop inside `fit()` by hand first, four lines at a time, and B.2 comes back to
this cell when the same loop has to run in mini-batches.

In [ ]:
def fit(model, X_train, y_train, X_val, y_val, epochs=5, lr=1e-3, batch=128, seed=42):
    """Train `model` with the four lines from Lecture 6 page 8 - forward, loss,
    backward, step - in mini-batches (Lecture 7 page 61), and record the
    train loss, validation loss and validation accuracy after every epoch.

        hist = fit(model, X_tr, y_tr, X_va, y_va, epochs=5)
        plot_curve(hist)

    X: float tensor (rows, features) or (rows, 28, 28) - whatever your model's
    first layer expects.  y: long tensor of class numbers 0..n_classes-1.
    Returns a DataFrame with one row per epoch. The curve is what you read;
    the last row is only where it stopped.
    """
    import torch, torch.nn as nn, pandas as _pd
    torch.manual_seed(seed)
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    model.to(dev)
    X_train, y_train, X_val, y_val = (t.to(dev) for t in (X_train, y_train, X_val, y_val))
    opt   = torch.optim.Adam(model.parameters(), lr=lr)      # Lecture 6 p.16: SGD, RMSprop, Adam
    lossf = nn.CrossEntropyLoss()                            # Lecture 6 p.10: softmax + cross-entropy
    rows  = []
    for ep in range(epochs):
        model.train()
        order, total = torch.randperm(len(X_train), device=dev), 0.0
        for i in range(0, len(X_train), batch):               # one mini-batch at a time (L06 p.22)
            idx  = order[i:i + batch]
            opt.zero_grad()
            loss = lossf(model(X_train[idx]), y_train[idx])  # forward, then the loss
            loss.backward()                                  # backward: gradient of the loss (L06 p.8)
            opt.step()                                       # step: move every weight (L07 p.60)
            total += loss.item() * len(idx)
        model.eval()
        with torch.no_grad():
            out = model(X_val)
            rows.append({'epoch': ep + 1, 'train_loss': total / len(X_train),
                         'val_loss': lossf(out, y_val).item(),
                         'val_acc': (out.argmax(1) == y_val).float().mean().item()})
        print('epoch %2d   train loss %.4f   val loss %.4f   val acc %.4f' % tuple(rows[-1].values()))
    return _pd.DataFrame(rows)


def accuracy(model, X, y):
    """Share of rows the model gets right. Works on any X the model accepts."""
    import torch
    dev = next(model.parameters()).device
    model.eval()
    with torch.no_grad():
        return (model(X.to(dev)).argmax(1) == y.to(dev)).float().mean().item()


def plot_curve(hist, title='train loss keeps falling; watch where val loss turns'):
    """The learning curve: train and validation loss against epoch, from fit()'s table."""
    import matplotlib.pyplot as _plt
    best = int(hist['val_loss'].idxmin())
    _plt.figure(figsize=(7, 4))
    _plt.plot(hist['epoch'], hist['train_loss'], color='0.6', label='train loss')
    _plt.plot(hist['epoch'], hist['val_loss'], color='C3', label='val loss')
    _plt.axvline(hist['epoch'][best], color='k', ls='--', lw=1, label='lowest val loss: epoch %d' % hist['epoch'][best])
    _plt.xlabel('epoch'); _plt.ylabel('cross-entropy loss'); _plt.title(title)
    _plt.legend(); _plt.tight_layout(); _plt.show()

print('fit(), accuracy() and plot_curve() ready')

### 🔵 A.1 — The neuron, and a loop you write once

**Lecture 6 page 5 draws one neuron:** every input times a weight, add them up, add a bias, pass the
sum through an activation function. Page 6 stacks them into a *dense layer* — every neuron connected
to every neuron in the layer before — and page 7 shows the activations; we use **ReLU**, the one
page 11 recommends for hidden nodes.

**The rows are session 1's.** The same titanic table, the same five leaky columns dropped, the same
preprocessing, the same 80/20 split with the same seed — so the tree from session 1 block C scores
exactly what it scored then. **First, the tree again, so its number is on screen:**

In [ ]:
df   = sns.load_dataset('titanic')
data = df.drop(columns=['alive', 'class', 'who', 'adult_male', 'deck'])   # session 1 C.1: leaky or duplicate
X, y = data.drop(columns=['survived']), data['survived']

NUM = ['age', 'sibsp', 'parch', 'fare', 'pclass']
CAT = ['sex', 'embarked', 'embark_town', 'alone']
preprocess = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')),
                      ('scale',  StandardScaler())]), NUM),
    ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')),
                      ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), CAT),
])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)

tree = Pipeline([('prep', preprocess), ('model', DecisionTreeClassifier(random_state=SEED))]).fit(X_train, y_train)
tree_acc = accuracy_score(y_test, tree.predict(X_test))
print('session 1 tree, test accuracy : %.4f' % tree_acc)

# the same columns, as tensors - the only new thing a network needs
X_tr = torch.tensor(preprocess.fit_transform(X_train), dtype=torch.float32)
X_te = torch.tensor(preprocess.transform(X_test),      dtype=torch.float32)
y_tr = torch.tensor(y_train.values); y_te = torch.tensor(y_test.values)
print('after preprocessing            :', tuple(X_tr.shape), 'train,', tuple(X_te.shape), 'test')

# Lecture 6 p.17: a validation set, carved out of the TRAINING rows. The test rows stay untouched.
X_a, X_v, y_a, y_v = train_test_split(X_tr, y_tr, test_size=0.2, random_state=SEED, stratify=y_tr)
print('rows the network trains on     :', len(X_a), '  validation:', len(X_v), '  test:', len(X_te))

✅ **Expected:** the tree at `0.8101` — session 1's number · `15` columns after one-hot encoding, `712`
train and `179` test rows · then `569` rows to train on, `143` for validation, the `179` test rows untouched.

**Why a validation set now, when sessions 1–3 had none:** Lecture 6 page 17 adds it for exactly one
job — deciding *how long to train*. A tree has no such decision. A network is trained one epoch at a
time, and something has to say when to stop that is not the test set.

**The network, in the words of the slides:**

| Lecture 6 says | In code |
|---|---|
| weighted sum + bias (p.5) · dense layer (p.6) | `nn.Linear(15, 64)` — 15 inputs, 64 neurons |
| activation (p.7) · ReLU for hidden nodes (p.11) | `nn.ReLU()` |
| output layer: one node per class, softmax, cross-entropy (p.10) | `nn.Linear(64, 2)` + `nn.CrossEntropyLoss()` |
| optimiser — SGD, RMSprop, Adam (p.16) · learning rate (p.11) | `torch.optim.Adam`, learning rate 0.01 |
| epoch (p.22) | one pass through the training rows |

*(Page 10 gives two classes a single sigmoid node with binary cross-entropy. Two softmax nodes are the
same model written so that the code never changes when the class count does — every network in this
file, including your own, uses that form.)*

**Then the four lines that are the whole of page 8** — forward, loss, backward, step — repeated. That
is the training loop, and it is written here by hand once:

In [ ]:
torch.manual_seed(SEED)
net = nn.Sequential(nn.Linear(15, 64), nn.ReLU(),
                    nn.Linear(64, 64), nn.ReLU(),
                    nn.Linear(64, 2))                       # two output nodes: died, survived
opt   = torch.optim.Adam(net.parameters(), lr=0.01)
lossf = nn.CrossEntropyLoss()

history = []
for epoch in range(300):                                    # 300 passes over the 569 rows
    net.train()
    opt.zero_grad()
    loss = lossf(net(X_a), y_a)                             # forward, then the loss     (L06 p.8)
    loss.backward()                                         # backward: gradient of the loss
    opt.step()                                              # step: every weight moves   (L07 p.60)

    net.eval()                                              # the same weights, scored on the validation rows
    with torch.no_grad():
        history.append({'epoch': epoch + 1, 'train_loss': loss.item(), 'val_loss': lossf(net(X_v), y_v).item()})
history = pd.DataFrame(history)

print('after 300 epochs: train loss %.4f   val loss %.4f' % (history.train_loss.iloc[-1], history.val_loss.iloc[-1]))
print('test accuracy, network : %.4f' % accuracy(net, X_te, y_te))
print('test accuracy, tree    : %.4f' % tree_acc)

✅ **Expected:** train loss `0.1687` · val loss `1.3044` · network `0.7877` · tree `0.8101`

*(All 569 rows go through in one step each epoch — Lecture 7 page 61 calls that batch gradient descent.
B.2 switches to mini-batches when the rows number 12,000.)*

**The loss fell to 0.17 and the network still lost to the tree by 2.2 points.** A loss that low says the
network fits the rows it trained on almost perfectly; the test accuracy says that did not carry over.
Two explanations, with opposite fixes: it trained *too little* and needs more epochs, or it trained
*too much* and memorised. **The two numbers printed above cannot tell them apart.** The third number in
`history` can.

### 🔵 A.2 — The curve that explains the number

**Lecture 6 draws it twice** — page 29 for iris, page 33 again: loss against epoch, one line for
training, one for validation. The loop in A.1 recorded both every epoch. 🆕 *Reading the pair as a
diagnosis — and acting on it — is the lab's addition; the slides show the picture and stop.*

In [ ]:
best = int(history.val_loss.idxmin())                       # the epoch where the validation loss was lowest
plot_curve(history, title='A.1: train loss keeps falling, val loss turns at epoch %d' % (best + 1))

print('lowest val loss %.4f at epoch %d' % (history.val_loss[best], best + 1))
print(history.iloc[[0, 9, best, 49, 99, 199, 299]].round(4).to_string(index=False))

# stop where the validation loss said to: the same network, trained for exactly that many epochs
torch.manual_seed(SEED)
net2 = nn.Sequential(nn.Linear(15, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 2))
opt2 = torch.optim.Adam(net2.parameters(), lr=0.01)
for epoch in range(best + 1):
    opt2.zero_grad(); loss = lossf(net2(X_a), y_a); loss.backward(); opt2.step()

print()
print('stopped at epoch %d : test accuracy %.4f   (train loss %.4f)' % (best + 1, accuracy(net2, X_te, y_te), loss.item()))
print('300 epochs          : test accuracy %.4f   (train loss %.4f)' % (accuracy(net, X_te, y_te), history.train_loss.iloc[-1]))
print('session 1 tree      : test accuracy %.4f' % tree_acc)

✅ **Expected:** the lowest val loss is `0.4449` at epoch `22` · after that the grey train line keeps
falling to `0.1687` while the red val line climbs to `1.3044` · stopped at 22 the network scores
`0.7989`; run to 300 it scores `0.7877`; the tree `0.8101`.

**Read the picture, not the last number.** Until epoch 22 both lines fall: the network is learning
something that also holds on rows it has not seen. After 22 only the train line falls — it is
memorising the 569 rows, and every memorised detail costs it on the 143 it was not shown. That is
*overfitting*, and Lecture 7 page 59 names the other cure, dropout; the cheapest one is just to stop.

**And the honest finding:** stopped at the right epoch the network gains a point — and still loses to
the tree. 891 rows is not a place where a neural network wins, and **"the network did not beat the
tree, here is the curve that shows I trained it properly" is a full answer.** Topic 13 in Part 2 puts
exactly that sentence in front of a group.

> **The rule for your hour:** print the curve before you trust the number. A loss alone cannot tell
> too-little from too-much; the pair can.

---
# B · From a Table to a Picture
🎤 **Walked live: all of B.** Lecture 7's convolution on the slide's own numbers, then on a real
picture — and the two-pixel test that shows what a dense layer never understood.

### 🔵 B.1 — Convolution, on the slide's own numbers

**Lecture 7 page 7 gives a 5×5 image and a 3×3 kernel.** Page 8 slides the kernel's centre onto
position (2,4), multiplies each weight by the pixel under it, adds the nine products, and gets **585**.
Pages 9–11 do it again with the kernel flipped and get **575** — that flip is the difference between
*cross-correlation* (page 6) and *convolution* (page 9). `torch` calls the page-8 version `conv2d`.

In [ ]:
I = torch.tensor([[17, 24,  1,  8, 15],
                  [23,  5,  7, 14, 16],
                  [ 4,  6, 13, 20, 22],
                  [10, 12, 19, 21,  3],
                  [11, 18, 25,  2,  9]], dtype=torch.float32)          # Lecture 7 p.7, the image
K = torch.tensor([[8, 1, 6],
                  [3, 5, 7],
                  [4, 9, 2]], dtype=torch.float32)                      # Lecture 7 p.7, the kernel

G = F.conv2d(I[None, None], K[None, None], padding=1)[0, 0]            # padding=1: zeros round the edge (L07 p.39)
print('page 8, cross-correlation at (2,4):', int(G[1, 3]))              # (2,4) on the slide is row 1, column 3 from zero

G_flip = F.conv2d(I[None, None], torch.flip(K, dims=(0, 1))[None, None], padding=1)[0, 0]
print('page 11, kernel flipped, at (2,4) :', int(G_flip[1, 3]))

print()
print('the whole output, page-8 version:')
print(G.int().numpy())

✅ **Expected:** `585` and `575` — the two numbers on Lecture 7 pages 8 and 11 — and a 5×5 output the
same size as the input, because `padding=1` put a ring of zeros round the image (page 39; the size
formula is page 40).

*(A network never cares about the flip: it learns the kernel, so whichever convention the layer uses it
learns the kernel that works under it. `conv2d` is the page-8 arithmetic and that is what every CNN below
runs.)*

**Now the same nine multiplications on a real picture.** Lecture 6 page 46 hands over two kernels and
calls what they do *feature extraction* — one lights up horizontal edges, one vertical. The pictures are
**Fashion-MNIST**: 28×28 grey clothes, ten classes, the same size and format as the MNIST digits on
Lecture 6 pages 35–37.

In [ ]:
from torchvision import datasets

fm_train = datasets.FashionMNIST('data', train=True,  download=True)     # 26 MB, once
fm_test  = datasets.FashionMNIST('data', train=False, download=True)
CLASSES  = fm_train.classes
print('train images:', len(fm_train), '  test images:', len(fm_test), '  one image:', tuple(fm_train.data[0].shape))
print('classes     :', CLASSES)

boot = fm_test.data[0].float() / 255.0                                    # L06 p.42: 0..255 -> 0..1
print('test image 0 is a', CLASSES[fm_test.targets[0]], '- pixel values now run', boot.min().item(), 'to', boot.max().item())

K_h = torch.tensor([[-1, -2, -1], [ 0, 0, 0], [ 1, 2, 1]], dtype=torch.float32)   # L06 p.46, horizontal edges
K_v = torch.tensor([[-1,  0,  1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32)   # L06 p.46, vertical edges
E_h = F.conv2d(boot[None, None], K_h[None, None], padding=1)[0, 0]
E_v = F.conv2d(boot[None, None], K_v[None, None], padding=1)[0, 0]

fig, ax = plt.subplots(1, 3, figsize=(10, 3.4))
for a, img, ttl in zip(ax, (boot, E_h, E_v), ('the boot, 28 x 28', 'page 46 kernel: horizontal edges', 'page 46 kernel: vertical edges')):
    a.imshow(img, cmap='gray'); a.set_title(ttl, fontsize=10); a.axis('off')
fig.tight_layout(); plt.show()

✅ **Expected:** `60000` train and `10000` test images of `28 × 28` · ten classes from T-shirt/top to
Ankle boot · image 0 is an **Ankle boot** with pixels rescaled to 0–1 · and three panels: the boot, then
its sole and top edge lit up by the horizontal kernel, then its heel and toe lit up by the vertical one.

**That is page 46's "feature extraction", with a kernel a person wrote.** Nine numbers, slid over the
whole picture, and the boot's outline falls out. B.2 asks the network to write its own nine numbers —
and first shows why it has to.

### 🔵 B.2 — Flatten it, then move it two pixels

**Lecture 6 page 39 and page 42 give the recipe for MNIST:** *flatten* the 28×28 grid into 784 numbers,
a dense layer of 128 ReLU neurons, ten softmax outputs — and 97.83% on the digits. The same recipe on
the clothes, with the loop from A.1 in mini-batches (`fit()` from the helper cell; Lecture 7 page 61,
Lecture 6 page 22). 12,000 training images and 2,000 for validation keep every cell under fifteen seconds.

In [ ]:
X_all = fm_train.data.float() / 255.0                        # (60000, 28, 28), 0..1
y_all = fm_train.targets
X_img_tr, y_img_tr = X_all[:12000], y_all[:12000]            # train on 12,000
X_img_va, y_img_va = X_all[12000:14000], y_all[12000:14000]  # validate on the next 2,000
X_img_te, y_img_te = fm_test.data.float() / 255.0, fm_test.targets

torch.manual_seed(SEED)
dense = nn.Sequential(nn.Flatten(),                          # L06 p.39: 28 x 28 -> 784
                      nn.Linear(784, 128), nn.ReLU(),        # L06 p.42: dense ReLU 128
                      nn.Linear(128, 10))                    # ten classes
hist_dense = fit(dense, X_img_tr, y_img_tr, X_img_va, y_img_va, epochs=5)

dense_acc = accuracy(dense, X_img_te, y_img_te)
print()
print('dense layer, test accuracy : %.4f    weights: %d' % (dense_acc, sum(p.numel() for p in dense.parameters())))

✅ **Expected:** five epoch lines, val accuracy climbing to about `0.84` · test accuracy `0.8349` ·
`101770` weights (784 × 128 + 128 + 128 × 10 + 10).

A respectable number — the page-42 recipe works on clothes as well as on digits. **Now the test nobody
runs:** the same 10,000 test pictures, every one moved two pixels to the right.

In [ ]:
def shift(X, dx=2):
    '''Move every image dx pixels to the right; the two columns that fall off are replaced by background.'''
    out = torch.zeros_like(X)
    out[:, :, dx:] = X[:, :, :-dx]
    return out

X_img_te_moved = shift(X_img_te, 2)
print('dense layer on the moved pictures: %.4f    (was %.4f)' % (accuracy(dense, X_img_te_moved, y_img_te), dense_acc))

fig, ax = plt.subplots(1, 2, figsize=(6, 3.2))
ax[0].imshow(X_img_te[0], cmap='gray');       ax[0].set_title('test image 0', fontsize=10)
ax[1].imshow(X_img_te_moved[0], cmap='gray'); ax[1].set_title('the same boot, 2 pixels right', fontsize=10)
for a in ax: a.axis('off')
fig.tight_layout(); plt.show()

✅ **Expected:** `0.5392` on the moved pictures, down from `0.8349` — **30 points lost to a shift you
can barely see** in the two panels.

**Why:** `Flatten` (page 39) turned position into identity. Pixel 300 and pixel 302 are two unrelated
inputs to a dense layer, each with its own weights, so a boot whose sole used to be at pixel 300 is now
a different set of numbers. The dense layer never learned *what a boot looks like*; it learned *which
pixels are bright when the label is boot*.

**Lecture 7 page 3:** a convolutional network *uses convolution in place of matrix multiplication*. The
kernel from B.1 slides over the whole picture, so the nine numbers that find a heel find it wherever the
heel is. Page 4 lists the four layers — convolution, ReLU, pooling, fully connected — and that is the
whole model below, trained by the same `fit()`:

In [ ]:
torch.manual_seed(SEED)
cnn = nn.Sequential(nn.Unflatten(1, (1, 28)),                            # (N, 28, 28) -> (N, 1 channel, 28, 28)
                    nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # L07 p.4: convolution -> ReLU -> pooling
                    nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # a second block: 28 -> 14 -> 7
                    nn.Flatten(), nn.Linear(16 * 7 * 7, 10))             # L07 p.57: fully connected, one node per class
hist_cnn = fit(cnn, X_img_tr, y_img_tr, X_img_va, y_img_va, epochs=5)

cnn_acc = accuracy(cnn, X_img_te, y_img_te)
print()
print('%-12s %-10s %-10s %s' % ('model', 'test', 'moved 2px', 'weights'))
print('%-12s %.4f     %.4f     %d' % ('dense', dense_acc, accuracy(dense, X_img_te_moved, y_img_te), sum(p.numel() for p in dense.parameters())))
print('%-12s %.4f     %.4f     %d' % ('CNN',   cnn_acc,   accuracy(cnn,   X_img_te_moved, y_img_te), sum(p.numel() for p in cnn.parameters())))

✅ **Expected**

| model | test | moved 2 px | weights |
|---|--:|--:|--:|
| dense | 0.8349 | 0.5392 | 101,770 |
| CNN | 0.8317 | 0.7395 | 9,098 |

**Same accuracy on the pictures as given, eleven times fewer weights, and a 9-point loss instead of 30
when the pictures move.** *(On the unmoved test set the dense layer is a third of a point ahead after five
epochs — say so; the CNN's advantage is not the first column, it is what the model stopped depending on.)*

Eight 3×3 kernels in the first layer, sixteen in the second: the network learned 24 small filters and
slides each over the whole image, exactly as B.1 slid the page-46 kernel. B.3 looks at what it wrote.

### 🔵 B.3 — What the network found

**Lecture 7 page 41 calls the output of a convolution layer a *feature map*, and Lecture 6 page 45
says the difference between machine learning and deep learning is that the feature extraction is
learned.** Here are the eight kernels the first layer learned, and the eight maps they make from the boot.

In [ ]:
with torch.no_grad():
    kernels = cnn[1].weight.detach()[:, 0]                       # (8, 3, 3): the eight learned kernels
    maps    = cnn[1](boot[None, None])[0].detach()               # (8, 28, 28): their eight feature maps on the boot

fig, ax = plt.subplots(2, 8, figsize=(13, 3.8))
for j in range(8):
    ax[0, j].imshow(kernels[j], cmap='RdBu_r', vmin=-1, vmax=1); ax[0, j].set_title('kernel %d' % j, fontsize=9)
    ax[1, j].imshow(maps[j], cmap='gray')
    for a in (ax[0, j], ax[1, j]): a.set_xticks([]); a.set_yticks([])
ax[0, 0].set_ylabel('3 x 3 weights', fontsize=9); ax[1, 0].set_ylabel('feature map', fontsize=9)
fig.suptitle('the first layer: eight kernels nobody wrote, and what each one sees in the boot', fontsize=10)
fig.tight_layout(); plt.show()

print('kernel 0, rounded:'); print(kernels[0].round(decimals=2).numpy())

✅ **Expected:** a top row of eight 3×3 colour grids (red positive, blue negative) and a bottom row of
eight versions of the boot — some a smoothed copy of it, some lit only along its top edge and sole, one
or two with the boot dark on a light ground. They differ from run to run only if the seed changes.

**Compare the bottom row with B.1's two edge panels.** Some of these maps are the page-46 kernels,
rediscovered; others are combinations no one would have written by hand. That is the sentence session 3
closed on — *the machine picks its own features* — and it is the whole reason a CNN survives the
two-pixel shift: a kernel that finds an edge finds it anywhere.

#### 📖 Read later — six curve shapes, and what to do about each

*Not walked. The page to keep open beside your own `plot_curve()`.* Every shape below is train loss
(grey) against validation loss (red), as in A.2.

| What the two lines do | What it means | What to do |
|---|---|---|
| both still falling at the last epoch | not finished | train longer — the number you have is not the number you would get |
| train falls, **val turns and climbs** (A.2) | overfitting: memorising rows | stop at the turn · dropout (Lecture 7 p.59) · more rows |
| both flat and high from the start | underfitting: the model cannot fit even the rows it sees | a bigger model · a higher learning rate (p.11 of Lecture 6 lists the usual values) |
| train jumps up and down, never settles | learning rate too high | divide it by 10 |
| both fall, but only just | learning rate too low | multiply it by 10, or run many more epochs |
| val *below* train | usually fine — the validation rows are easier, or the split is tiny | say so; do not "fix" it |

**Save what you trained** so the curve and the number stay attached to real weights:
`torch.save(model.state_dict(), 'model.pt')` and later `model.load_state_dict(torch.load('model.pt'))`.

#### 🎨 Read later — polishing the chart, rung 4 of 6: *make the one line you will point at the only one in colour*

Session 1 said the title is the conclusion; session 2 said fewer marks; session 3 said pick the chart
from the question. **This session's rung is about a chart you will stand in front of: the learning
curve.** Look at A.2's figure again — the train line is grey, the val line is the one colour, the epoch
that decided the answer is the one dashed mark, and the title says what happened rather than "loss vs
epoch". `plot_curve()` already does that; when you draw your own, keep the rule:

1. **Everything you are not talking about is grey.** One colour for the one line the sentence is about.
2. **One mark for the decision** — a dashed line at the epoch you chose, with the number in the legend.
3. **Title = the finding.** *"val loss turns at epoch 22"*, not *"training curve"*.

---
---
# 📋 Part 2 — Pick Your Topic
### Assignment 4 · one of these 20 · this takes ~8 minutes

**Groups of three or four. One topic per group, first come first served, no two groups on the same one.**

**Every topic here is classification** — a label per row, pictures or a table — and every one can be
trained from scratch in your hour on a plain CPU. Take yours and go the whole way:

```
your data -> EDA -> prepare (0..1 for pixels, scale for tables, validation split) -> baseline() + a tree -> a network with fit() -> the curve -> the number
```

📄 **[How it is marked, and what to hand in](https://classes.incortx.com/DataAnalytics/session-00/)** — the short version: you need **a baseline beside
every number**, **the model you already know beside the network**, and **a learning curve with the
epoch you chose marked on it**. A network that loses to the tree still scores full marks if the curve
shows you trained it properly.

**The 20 topics.** 🟢 straightforward · 🟡 has something awkward in it

The **trap** column is not a general warning. It is the specific thing that will bite you in that
dataset, and it is where the questions will come from. Eleven topics are pictures and nine are tables —
a table gets the A block's network, a picture gets B's, and a picture group should run the two-pixel test.

| # | Topic | Data | The trap | |
|:--:|---|---|---|:--:|
| **1** | Read handwritten digits | `torchvision.datasets.MNIST` (60,000 train + 10,000 test, 28x28 grey) | The benchmark everyone knows, and the worked answer in Part 3 -- Lecture 6 p.42 reports **97.83%** with the dense recipe on all 60,000; on 12,000 rows and 5 epochs you get 0.9281. **Say why your number is lower before anyone asks** | 🟢 |
| **2** | Read ten Japanese characters | `torchvision.datasets.KMNIST` (60,000 train, 28x28 grey, exactly 6,000 per class) | The same shape as MNIST, but **you cannot tell a mistake by eye** -- the ten characters are unfamiliar, so the confusion matrix has to do the judging that your eyes did on digits | 🟡 |
| **3** | Read 26 handwritten letters | `torchvision.datasets.EMNIST` (`split="letters"`, 124,800 train, 4,800 per letter) | **Labels run 1 to 26, not 0 to 25** -- `CrossEntropyLoss` crashes on label 26 until you subtract one. And every image is stored transposed: `.transpose(1, 2)` before you look, or the w's are 3's | 🟡 |
| **4** | Read digits from 16x16 postal scans | `torchvision.datasets.USPS` (7,291 train, 16x16 grey) | **Sixteen pixels across, so two poolings leave 4x4** -- B.2's `nn.Linear(16 * 7 * 7, 10)` is wrong here; size it from Lecture 7 p.40's formula. Classes run from 542 to 1,194 rows | 🟢 |
| **5** | Classify land use from satellite tiles | `torchvision.datasets.EuroSAT` (27,000 tiles of 64x64 colour, 10 classes of 2,000-3,000) | **No train/test split comes with it** -- you make one, stratified. And 64x64 colour is 12,288 inputs to a flatten; resize to 32x32 with a transform or the dense layer alone has 1.5 million weights | 🟡 |
| **6** | Recognise 43 German traffic signs | `torchvision.datasets.GTSRB` (26,640 train, colour, sizes vary) | **Classes run from 150 to 1,500 images** and the pictures come in different sizes -- a `Resize` transform is not optional, and accuracy hides the ten signs with 150 rows; look at them in the confusion matrix | 🟡 |
| **7** | Read 8x8 digits with a network | `sklearn.datasets.load_digits()` (1,797 images of 8x8) | **Pixels run 0 to 16, not 0 to 255** -- divide by 16. And 8x8 survives only one pooling. Session 1's tree gets 0.8222 here; the smallest set in the bank, so the tree may not lose | 🟢 |
| **8** | Is this chest X-ray pneumonia? | `zenodo.org/records/10519652/files/pneumoniamnist.npz` (4,708 train, 28x28 grey; split included) | **74.2% of the training rows are pneumonia**, so `baseline()` scores 0.742 by always saying yes -- and a missed pneumonia is the expensive mistake. Accuracy is the wrong number; say which one you used instead | 🟡 |
| **9** | Which of seven skin conditions is this? | `zenodo.org/records/10519652/files/dermamnist.npz` (7,007 train, 28x28 colour, 7 classes) | **67% of rows are one class** (nevus) and four classes have under 5% each -- a five-epoch network never predicts the small ones and still scores 0.67. Per-class recall, or the number means nothing | 🟡 |
| **10** | Which of eight blood cell types is this? | `zenodo.org/records/10519652/files/bloodmnist.npz` (11,959 train, 28x28 colour, 8 classes of 7-19%) | Colour: the array is `(N, 28, 28, 3)`, so **`.permute(0, 3, 1, 2)` and a first `Conv2d(3, ...)`** -- every other line of B.2's CNN stays. The cleanest medical set here | 🟢 |
| **11** | Which organ is in this CT slice? | `zenodo.org/records/10519652/files/organamnist.npz` (34,561 train, 28x28 grey, 11 classes) | **Eleven classes from 3.9% to 17.8% of rows** -- and the slices of neighbouring organs look alike. The confusion matrix will have a few large off-diagonal cells; name them | 🟡 |
| **12** | Does this person earn over 50K? | `archive.ics.uci.edu/static/public/2/adult.zip` (`adult.data`, 32,561 rows, no header) | Missing values are written as ` ?` with a leading space -- `na_values='?', skipinitialspace=True`. 76% earn under 50K, so `baseline()` is 0.76; session 1's tree gets 0.8130 and the network has to beat *that* | 🟢 |
| **13** | Which species is this penguin? -- the tree wins | `sns.load_dataset('penguins')` (333 complete rows, 3 species) | **A.2's sentence, handed to a group:** the tree scores 0.9254 on 67 test rows, and a network on 266 training rows will not beat it reliably. The honest write-up is the curve that shows you trained it properly, and the tree's number on top | 🟢 |
| **14** | Will this customer subscribe? | `archive.ics.uci.edu/static/public/222/bank+marketing.zip` (a zip inside a zip: `bank.zip` -> `bank-full.csv`, `sep=";"`, 45,211 rows) | **`duration` is a leak** -- the length of the call whose outcome is the target (session 1 C.3): the tree scores 0.8728 with it and 0.8309 without. 11.7% say yes, so accuracy alone is also wrong | 🟡 |
| **15** | Is this tumour malignant? -- scale, or the network stalls | `sklearn.datasets.load_breast_cancer()` (569 rows, 30 columns) | **`worst area` runs to 4,254 while `smoothness` tops out at 0.03** -- unscaled, a network sits at the majority class (0.6316, measured with sklearn's MLP); scaled it reaches 0.9561. Session 3 A.3 again, and this time it stops the model learning at all | 🟡 |
| **16** | Recognise letters from 16 measured features | `archive.ics.uci.edu/static/public/59/letter+recognition.zip` (`letter-recognition.data`, 20,000 rows, no header) | **The one table here where the network clearly wins** -- the tree gets 0.8820, a scaled 128-neuron network 0.9600 (measured with sklearn's MLP). 26 classes of 734-813 rows; the letter is the first column | 🟢 |
| **17** | Is this e-mail spam? | `archive.ics.uci.edu/static/public/94/spambase.zip` (`spambase.data`, 4,601 rows, no header) | **The last three columns run to 15,841 while the first 48 top out at 42.81** -- scale, or those three are the only inputs the network hears. 39.4% spam; the tree gets 0.9110 | 🟡 |
| **18** | Gamma ray or hadron shower? | `archive.ics.uci.edu/static/public/159/magic+gamma+telescope.zip` (`magic04.data`, 19,020 rows, no header) | **64.8% gamma** and, in the original problem, a hadron mistaken for a gamma is the costly error -- per-class recall, not accuracy. The tree gets 0.8186 | 🟡 |
| **19** | Which forest cover type is this? | `sklearn.datasets.fetch_covtype()` (581,012 rows, 54 columns, 7 classes) | **Big enough that batch size and training time become decisions** -- 581,012 rows is 4,540 mini-batches per epoch; take a subset and say how big. And class 4 is 0.47% of rows | 🟡 |
| **20** | Is this mushroom edible? | `archive.ics.uci.edu/static/public/73/mushroom.zip` (`agaricus-lepiota.data`, 8,124 rows, all columns categorical) | **`odor` alone gives the tree 0.9858 and all columns give 1.0000** -- a network is not needed, and a group that says so with the numbers has understood the session. Every column is text: one-hot all 22 | 🟡 |

In [ ]:
# ── Record your group's choice ─────────────────────────────────────────────
TOPIC_ID = None      # <- put your group's topic number here, then run this cell

TOPIC_TRAPS = {
     1: ('Read handwritten digits',
        'The benchmark everyone knows, and the worked answer in Part 3 -- Lecture 6 p.42 reports **97.83%** with the dense recipe on all 60,000; on 12,000 rows and 5 epochs you get 0.9281. **Say why your number is lower before anyone asks**'),
     2: ('Read ten Japanese characters',
        'The same shape as MNIST, but **you cannot tell a mistake by eye** -- the ten characters are unfamiliar, so the confusion matrix has to do the judging that your eyes did on digits'),
     3: ('Read 26 handwritten letters',
        "**Labels run 1 to 26, not 0 to 25** -- `CrossEntropyLoss` crashes on label 26 until you subtract one. And every image is stored transposed: `.transpose(1, 2)` before you look, or the w's are 3's"),
     4: ('Read digits from 16x16 postal scans',
        "**Sixteen pixels across, so two poolings leave 4x4** -- B.2's `nn.Linear(16 * 7 * 7, 10)` is wrong here; size it from Lecture 7 p.40's formula. Classes run from 542 to 1,194 rows"),
     5: ('Classify land use from satellite tiles',
        '**No train/test split comes with it** -- you make one, stratified. And 64x64 colour is 12,288 inputs to a flatten; resize to 32x32 with a transform or the dense layer alone has 1.5 million weights'),
     6: ('Recognise 43 German traffic signs',
        '**Classes run from 150 to 1,500 images** and the pictures come in different sizes -- a `Resize` transform is not optional, and accuracy hides the ten signs with 150 rows; look at them in the confusion matrix'),
     7: ('Read 8x8 digits with a network',
        "**Pixels run 0 to 16, not 0 to 255** -- divide by 16. And 8x8 survives only one pooling. Session 1's tree gets 0.8222 here; the smallest set in the bank, so the tree may not lose"),
     8: ('Is this chest X-ray pneumonia?',
        '**74.2% of the training rows are pneumonia**, so `baseline()` scores 0.742 by always saying yes -- and a missed pneumonia is the expensive mistake. Accuracy is the wrong number; say which one you used instead'),
     9: ('Which of seven skin conditions is this?',
        '**67% of rows are one class** (nevus) and four classes have under 5% each -- a five-epoch network never predicts the small ones and still scores 0.67. Per-class recall, or the number means nothing'),
    10: ('Which of eight blood cell types is this?',
        "Colour: the array is `(N, 28, 28, 3)`, so **`.permute(0, 3, 1, 2)` and a first `Conv2d(3, ...)`** -- every other line of B.2's CNN stays. The cleanest medical set here"),
    11: ('Which organ is in this CT slice?',
        '**Eleven classes from 3.9% to 17.8% of rows** -- and the slices of neighbouring organs look alike. The confusion matrix will have a few large off-diagonal cells; name them'),
    12: ('Does this person earn over 50K?',
        "Missing values are written as ` ?` with a leading space -- `na_values='?', skipinitialspace=True`. 76% earn under 50K, so `baseline()` is 0.76; session 1's tree gets 0.8130 and the network has to beat *that*"),
    13: ('Which species is this penguin? -- the tree wins',
        "**A.2's sentence, handed to a group:** the tree scores 0.9254 on 67 test rows, and a network on 266 training rows will not beat it reliably. The honest write-up is the curve that shows you trained it properly, and the tree's number on top"),
    14: ('Will this customer subscribe?',
        '**`duration` is a leak** -- the length of the call whose outcome is the target (session 1 C.3): the tree scores 0.8728 with it and 0.8309 without. 11.7% say yes, so accuracy alone is also wrong'),
    15: ('Is this tumour malignant? -- scale, or the network stalls',
        "**`worst area` runs to 4,254 while `smoothness` tops out at 0.03** -- unscaled, a network sits at the majority class (0.6316, measured with sklearn's MLP); scaled it reaches 0.9561. Session 3 A.3 again, and this time it stops the model learning at all"),
    16: ('Recognise letters from 16 measured features',
        "**The one table here where the network clearly wins** -- the tree gets 0.8820, a scaled 128-neuron network 0.9600 (measured with sklearn's MLP). 26 classes of 734-813 rows; the letter is the first column"),
    17: ('Is this e-mail spam?',
        '**The last three columns run to 15,841 while the first 48 top out at 42.81** -- scale, or those three are the only inputs the network hears. 39.4% spam; the tree gets 0.9110'),
    18: ('Gamma ray or hadron shower?',
        '**64.8% gamma** and, in the original problem, a hadron mistaken for a gamma is the costly error -- per-class recall, not accuracy. The tree gets 0.8186'),
    19: ('Which forest cover type is this?',
        '**Big enough that batch size and training time become decisions** -- 581,012 rows is 4,540 mini-batches per epoch; take a subset and say how big. And class 4 is 0.47% of rows'),
    20: ('Is this mushroom edible?',
        '**`odor` alone gives the tree 0.9858 and all columns give 1.0000** -- a network is not needed, and a group that says so with the numbers has understood the session. Every column is text: one-hot all 22'),
}

if TOPIC_ID in TOPIC_TRAPS:
    title, trap = TOPIC_TRAPS[TOPIC_ID]
    print('Topic %d: %s' % (TOPIC_ID, title))
    print('Watch out for : %s' % trap)
else:
    print('Set TOPIC_ID to your group number (1-20) and run this cell again.')

✅ **Expected:** your topic and its trap printed back at you. Write the trap somewhere you will see it
again — it is the first thing to check when your results look strange.

---
---
# 🟠 Part 3 — Your Hour · 60 Minutes, Your Own Data

**Everything above was the demo.** From here it is your group's work, and it is what gets marked.

This section does not depend on a single cell above it. Run it from the top of this section and it works.

### The steps, and the clock

| Minutes | Step | What has to exist when you are done |
|:--:|---|---|
| — | **0 · Data already loaded** | you did this before class. `shape` · a few rows or pictures · one sentence on what a row is |
| 0–12 | **1 · EDA → one insight** | a chart, and a sentence stating what you *found* — class balance is the one to look at first |
| 12–27 | **2 · Prepare the data** | pixels to 0–1 or columns scaled, a **validation split** out of the training rows, tensors — **and why** |
| 27–37 | **3 · Metric + baseline** | accuracy (F1 if the classes are skewed — session 1) · `baseline()` · **and the session-1 tree on the same rows** |
| 37–50 | **4 · Today's technique** *(if you get there)* | a network with `fit()`, its curve, the epoch you chose and why · a CNN if your rows are pictures |
| 50–60 | **5 · Write up + get ready** | one sentence on the finding, one on what you do not trust, notebook scrolled to where you start |

**Steps 1, 2 and 3 are what you present and what is marked. Step 4 is a bonus.**

**Three sets this session, not two:** the validation rows come out of the training rows (Lecture 6
page 17) and decide how long to train. The test rows are touched once, at the end, for the number.

In [ ]:
# ── SUBMISSION HEADER — fill this in first ─────────────────────────────────
GROUP     = ''            # your group letter: 'A' .. 'J'
MEMBERS   = ['', '', '']  # everyone in the group - keep this order all term
TOPIC_ID  = None          # the topic number your group claimed

# EVERY member speaks in the video. Every assignment, no exceptions.
IN_CLASS  = ['', '']      # the TWO representing the group in the room

# ── check ───────────────────────────────────────────────────────────────────
_all   = [m.strip() for m in MEMBERS  if m.strip()]
_room  = [m.strip() for m in IN_CLASS if m.strip()]

print(f'Group {GROUP or "?"} | topic {TOPIC_ID} | {len(_all)} members')
print(f'  in the room  : {", ".join(_room) or "-- nobody --"}')
print(f'  in the video : everyone - {", ".join(_all) or "-- nobody --"}')

unknown = [m for m in _room if m not in _all]
if not GROUP or not _all or TOPIC_ID is None:
    print('\n[ ] header not filled in yet')
elif unknown:
    print(f'\n[!] not found in MEMBERS: {", ".join(unknown)} - check the spelling')
elif len(_room) != 2:
    print(f'\n[!] {len(_room)} named for the room, should be exactly 2')
else:
    print('\n[ok] two representatives named, and every member speaks in the video')

### 🟠 Setup for this section

Its own imports and its own helpers, so this half runs whatever happened above.

In [ ]:
import warnings; warnings.filterwarnings('ignore')

import io as _io, zipfile, urllib.request        # for the UCI zips and the MedMNIST files
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torchvision import datasets, transforms

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
pd.set_option('display.width', 120); pd.set_option('display.max_columns', 30)
plt.rcParams['figure.figsize'] = (7, 4)
plt.rcParams['axes.grid'] = True; plt.rcParams['grid.alpha'] = 0.3
print('torch', torch.__version__, '| GPU:', torch.cuda.is_available(), '- ready')

**Four helpers come with the notebook: `eda()`, `baseline()`, `fit()` and `plot_curve()`** (plus
`accuracy()`). Run the cell below once.

**None of them does the thinking.** `eda()` prints the six views; `baseline()` prints the do-nothing
score; `fit()` is the loop from B.2 and returns the table `plot_curve()` draws. **Reading them is the
marked part.**

In [ ]:
def eda(df, target=None, n=6):
    """Print the six things you should always look at first. Reading them is your job.

        eda(df)                    # no target column yet
        eda(df, target='outcome')  # adds class balance + correlation with the target
    """
    import pandas as _pd
    line = '\u2500' * 62

    print(line); print(f'1. SHAPE      {df.shape[0]:,} rows x {df.shape[1]} columns')
    print(line); print('2. ONE ROW    what does a single row actually represent?')
    print(df.head(3).to_string())

    print(line); print('3. TYPES      a number stored as text will not go into a model')
    info = _pd.DataFrame({'dtype': df.dtypes.astype(str),
                          'non_null': df.notna().sum(),
                          'distinct': df.nunique()})
    print(info.to_string())

    print(line); print('4. MISSING    how much, and in which columns')
    miss = df.isna().sum()
    miss = miss[miss > 0].sort_values(ascending=False)
    if len(miss) == 0:
        print('   isna() finds none  --  but a 0 or a -200 can be a missing value in disguise.')
        print('   Check step 5 for impossible values before you believe this.')
    else:
        print(_pd.DataFrame({'missing': miss, 'percent': (miss / len(df) * 100).round(1)}).to_string())

    print(line); print('5. RANGES     look for a min or max that cannot be real')
    num = df.select_dtypes('number')
    print(num.describe().T[['min', '25%', '50%', '75%', 'max']].to_string() if len(num.columns)
          else '   no numeric columns')

    if target is not None and target in df.columns:
        print(line); print(f'6. TARGET     {target!r}')
        y = df[target]
        if y.dtype.kind == 'f' or y.nunique() > 20:
            print(y.describe().to_string())
            corr = num.corr(numeric_only=True)[target].drop(target).sort_values(key=abs, ascending=False)
            print(f'\n   strongest correlations with {target}:')
            print(corr.head(n).round(4).to_string())
        else:
            print(y.value_counts(normalize=True).round(4).to_string())
            print(f'\n   most common class is {y.value_counts(normalize=True).max():.1%} of rows')
    print(line)
    print('Now write ONE sentence about something you did not know 60 seconds ago.')

def baseline(y_train, y_test, kind='auto'):
    """Print the score of the dumbest possible model for this target.

    You do not have to write a baseline. You do have to read it and say what it
    means -- that is the part that carries marks.

        baseline(y_tr, y_te)            # works out classification vs regression
        baseline(y_tr, y_te, 'clf')     # force it
        baseline(y_tr, y_te, 'reg')
    """
    import numpy as _np, pandas as _pd
    from sklearn.dummy import DummyClassifier, DummyRegressor
    from sklearn.metrics import (accuracy_score, f1_score, mean_absolute_error,
                                 mean_squared_error, r2_score)

    ytr, yte = _pd.Series(y_train), _pd.Series(y_test)
    if kind == 'auto':
        numeric = _pd.api.types.is_numeric_dtype(ytr)
        kind = 'reg' if (numeric and (ytr.dtype.kind == 'f' or ytr.nunique() > 20)) else 'clf'
        looks = 'a number -> regression' if kind == 'reg' else 'a label -> classification'
        print(f'[auto] your target looks like {looks}'
              f'  ({ytr.nunique()} distinct values, dtype {ytr.dtype})')
        print("       wrong guess? pass kind='clf' or kind='reg'")

    Xtr = _np.zeros((len(ytr), 1))          # a baseline ignores the features on purpose
    Xte = _np.zeros((len(yte), 1))

    if kind == 'clf':
        m = DummyClassifier(strategy='most_frequent').fit(Xtr, ytr)
        p = m.predict(Xte)
        top = m.predict(Xte[:1])[0]          # the class it actually predicts
        top = top.item() if hasattr(top, 'item') else top
        acc = accuracy_score(yte, p)
        f1 = f1_score(yte, p, average='binary' if yte.nunique() == 2 else 'macro',
                      zero_division=0)
        print(f'baseline = always predict the most common class ({top!r})')
        print(f'  accuracy {acc:.4f}')
        print(f'  F1       {f1:.4f}   <- same model. If these two disagree, accuracy is the wrong metric')
        return {'accuracy': acc, 'f1': f1}

    m = DummyRegressor(strategy='median').fit(Xtr, ytr)
    p = m.predict(Xte)
    mae = mean_absolute_error(yte, p)
    rmse = mean_squared_error(yte, p) ** 0.5
    r2 = r2_score(yte, p)
    print(f'baseline = always predict the training median ({_np.median(ytr):.4f})')
    print(f'  MAE  {mae:.4f}')
    print(f'  RMSE {rmse:.4f}')
    print(f'  R2   {r2:.4f}   <- a baseline R2 at or just below zero is correct, not a bug')
    return {'mae': mae, 'rmse': rmse, 'r2': r2}

def fit(model, X_train, y_train, X_val, y_val, epochs=5, lr=1e-3, batch=128, seed=42):
    """Train `model` with the four lines from Lecture 6 page 8 - forward, loss,
    backward, step - in mini-batches (Lecture 7 page 61), and record the
    train loss, validation loss and validation accuracy after every epoch.

        hist = fit(model, X_tr, y_tr, X_va, y_va, epochs=5)
        plot_curve(hist)

    X: float tensor (rows, features) or (rows, 28, 28) - whatever your model's
    first layer expects.  y: long tensor of class numbers 0..n_classes-1.
    Returns a DataFrame with one row per epoch. The curve is what you read;
    the last row is only where it stopped.
    """
    import torch, torch.nn as nn, pandas as _pd
    torch.manual_seed(seed)
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    model.to(dev)
    X_train, y_train, X_val, y_val = (t.to(dev) for t in (X_train, y_train, X_val, y_val))
    opt   = torch.optim.Adam(model.parameters(), lr=lr)      # Lecture 6 p.16: SGD, RMSprop, Adam
    lossf = nn.CrossEntropyLoss()                            # Lecture 6 p.10: softmax + cross-entropy
    rows  = []
    for ep in range(epochs):
        model.train()
        order, total = torch.randperm(len(X_train), device=dev), 0.0
        for i in range(0, len(X_train), batch):               # one mini-batch at a time (L06 p.22)
            idx  = order[i:i + batch]
            opt.zero_grad()
            loss = lossf(model(X_train[idx]), y_train[idx])  # forward, then the loss
            loss.backward()                                  # backward: gradient of the loss (L06 p.8)
            opt.step()                                       # step: move every weight (L07 p.60)
            total += loss.item() * len(idx)
        model.eval()
        with torch.no_grad():
            out = model(X_val)
            rows.append({'epoch': ep + 1, 'train_loss': total / len(X_train),
                         'val_loss': lossf(out, y_val).item(),
                         'val_acc': (out.argmax(1) == y_val).float().mean().item()})
        print('epoch %2d   train loss %.4f   val loss %.4f   val acc %.4f' % tuple(rows[-1].values()))
    return _pd.DataFrame(rows)


def accuracy(model, X, y):
    """Share of rows the model gets right. Works on any X the model accepts."""
    import torch
    dev = next(model.parameters()).device
    model.eval()
    with torch.no_grad():
        return (model(X.to(dev)).argmax(1) == y.to(dev)).float().mean().item()


def plot_curve(hist, title='train loss keeps falling; watch where val loss turns'):
    """The learning curve: train and validation loss against epoch, from fit()'s table."""
    import matplotlib.pyplot as _plt
    best = int(hist['val_loss'].idxmin())
    _plt.figure(figsize=(7, 4))
    _plt.plot(hist['epoch'], hist['train_loss'], color='0.6', label='train loss')
    _plt.plot(hist['epoch'], hist['val_loss'], color='C3', label='val loss')
    _plt.axvline(hist['epoch'][best], color='k', ls='--', lw=1, label='lowest val loss: epoch %d' % hist['epoch'][best])
    _plt.xlabel('epoch'); _plt.ylabel('cross-entropy loss'); _plt.title(title)
    _plt.legend(); _plt.tight_layout(); _plt.show()

print('eda(), baseline(), fit(), accuracy() and plot_curve() ready')

### 🟠 Opening your data — three shapes this session

**Pictures from `torchvision` (topics 1–6).** One line downloads, and the MNIST family (topics 1–4)
keeps the pixels in `.data` and the labels in `.targets`:

```python
ds = datasets.KMNIST('data', train=True, download=True)          # topics 1-4: MNIST · KMNIST · EMNIST · USPS
X, y = ds.data.float() / 255.0, ds.targets                        # (N, 28, 28) in 0..1, and class numbers
```

EuroSAT and GTSRB (topics 5, 6) hand out one colour picture at a time, in more than one size, so give
them a transform and stack:

```python
tf = transforms.Compose([transforms.Resize((32, 32)), transforms.ToTensor()])   # (3, 32, 32) in 0..1
ds = datasets.EuroSAT('data', download=True, transform=tf)
X  = torch.stack([ds[i][0] for i in range(len(ds))]); y = torch.tensor([ds[i][1] for i in range(len(ds))])
```

Topic 7 is `sklearn.datasets.load_digits()`: `torch.tensor(d.images).float() / 16.0` — its pixels run
0–16, not 0–255.

**Pictures from MedMNIST (topics 8–11)** are one `.npz` file each, with the split already made:

```python
with urllib.request.urlopen(URL) as r: z = np.load(_io.BytesIO(r.read()))
X_tr = torch.tensor(z['train_images']).float() / 255.0; y_tr = torch.tensor(z['train_labels']).squeeze().long()
```

**Tables (topics 12–20)** load exactly as in sessions 1–3 — `sns.load_dataset`, `sklearn.datasets`, or a
UCI zip with `zipfile` — and then go through the session-1 `ColumnTransformer` before they become a
tensor: `torch.tensor(preprocess.fit_transform(X_train), dtype=torch.float32)`. Topic 14 (bank marketing)
is a zip inside a zip; topic 12 (adult) writes missing values as ` ?` with a space.

A colour picture is `(N, 3, H, W)` and the first `Conv2d` takes 3 channels instead of 1; every other
line of B.2's CNN stays. If the pictures are not 28×28, print the shape after the two poolings before
you size the final `Linear` (Lecture 7 page 40 has the formula).

### 🟠 Step 0 — Your data, already loaded

Load it, then look at it the way the demo did: **how many rows, what one row is, and how the classes
are shared out.** Pictures do not go into `eda()` — print `X.shape` and a grid of a few instead. Tables
do: `eda(df, target='...')`.

In [ ]:
# TODO: load your group's data. Pictures: X (N, H, W) or (N, C, H, W) in 0..1 and y as class numbers.
#       Tables: df, then eda(df, target=...).

✅ **What topic 1 prints in under a second**

- **`60000` train and `10000` test images of `28 × 28`**, pixels already in 0–1 after the divide.
- **Rows per class:** train from `5421` (the 5s) to `6742` (the 1s); test from `892` to `1135`. Ten classes,
  none rare — so accuracy is a fair metric here. That is not true of topics 8, 9, 12 or 14; look before
  you choose.

**Write the one sentence:** what is one row of this data?

### 🟠 Step 1 — EDA → one insight · *0–12 min*

**The chart is not the deliverable. The sentence under it is.**

For pictures the chart that earns its place is **the class balance as bars** — it decides the metric —
and **a grid of examples from the classes you expect to confuse**. For a table it is what it was in
session 1: the target's balance, and the one column that separates the classes best.

| Not an insight | An insight |
|---|---|
| "This is a bar chart of the classes." | "Class 1 has 26% more rows than class 5 — mild, so accuracy is fair; F1 goes beside it anyway." |
| "Here are some images." | "The 4s and 9s share a shape; if the confusion matrix has one big off-diagonal cell it will be there." |

In [ ]:
# TODO: one chart that shows something about your classes or your pictures

✅ **Expected on topic 1:** the 1s have **24%** more rows than the 5s (`6742` against `5421`) — mild, so
accuracy is a fair metric; F1 goes beside it anyway. And the strip: a closed loop is the only difference
between a 4 and a 9 on several of these.

**What I found:** *(one sentence — a finding, not a description of the chart)*

**What this changes about what I do next:** *(write it here)*

### 🟠 Step 2 — Prepare the data · *12–27 min*

Everything step 1 told you was wrong, you now fix — **and you write down why you fixed it that way.**

**Three rules this session:**

1. **Pixels to 0–1, columns scaled.** Lecture 6 page 42 divides by 255; pages 26–27 standardise a
   table. A network that gets raw `worst area` next to `smoothness` (topic 15) stalls at the majority
   class — it is A.3 of session 3 again, and this time it stops the model learning at all.
2. **A validation split, out of the training rows** (Lecture 6 page 17). The test rows are for the
   number at the end and nothing else.
3. **Tensors:** `float32` for the inputs, `long` (class numbers 0..k−1) for the labels. A string label
   has to become a number first — `pd.factorize` or `LabelEncoder`.

> **The tree needs the flat, scaled table too** — reshape pictures to `(N, 784)` for it. It ignores the
> scale, but it needs the same rows the network gets or the comparison in step 3 is not fair.

In [ ]:
# TODO: pixels to 0..1 (or a ColumnTransformer for a table), a validation split, tensors - and a comment saying why

**What I fixed, and why I chose that fix:** *(one line per decision)*

### 🟠 Step 3 — Metric + baseline · *27–37 min*

**Two lines under every number this session, not one.** `baseline()` is the do-nothing score from
session 1. The second line is **the model you already know — a decision tree on the same rows** —
because the question the whole session asks is *did the network earn its place*. A.2 answered "no" on
titanic; your topic gets its own answer.

**Two rules that still belong to you:**

1. **Pick the metric from the class balance** (step 1). Skewed classes: F1 beside accuracy, as in
   session 1 C.4.
2. **The tree gets the same training rows and the same test rows** as the network will.

In [ ]:
# TODO: baseline(y_tr, y_test), then a DecisionTreeClassifier on the same rows - print both

✅ **Expected on topic 1:** `baseline()` says always-predict-1 scores `0.1135` accuracy and `0.0204` F1 ·
the tree on raw pixels scores **`0.8204`**.

**Our metric is ___ because ___** *(fill this in — it is half of what requirement 1 looks for)*

**The number the network has to beat is ___, not the baseline** *(write the tree's number here)*

### 🟠 Step 4 — Today's technique · *37–50 min* — **if you get there**

**This step is a bonus, not a requirement.** The B.2 recipe with `fit()`: a dense network first, its
curve, then — if your rows are pictures — the CNN, and the two-pixel test on both. One table at the end:
baseline, tree, dense, CNN, with the epoch you stopped at and why.

In [ ]:
# TODO: a dense network with fit() -> plot_curve(); then the CNN if your rows are pictures; one table of every number

✅ **Expected on topic 1**

| model | test accuracy | moved 2 px |
|---|--:|--:|
| baseline (always 1) | 0.1135 | — |
| tree, 784 pixels | 0.8204 | — |
| dense, 5 epochs | 0.9281 | 0.6766 |
| CNN, 5 epochs | 0.9427 | 0.8502 |

Both curves are still falling at epoch 5 — **stopped early, not overfitted**: the honest sentence is
"five epochs was the clock, not the curve; the number would rise with more". The dense layer beats the
tree by 11 points, the CNN by 12 — and on the moved digits the CNN keeps 0.85 where the dense layer
drops to 0.68, the B.2 result again on a different set.

### 🟠 Step 5 — Write it up and close the file properly · *50–60 min*

Two sentences per group, both short. **You are not presenting today**, so the rest of this goes into
making the file usable when you reopen it at home:

1. **State the finding with all three numbers in it** — baseline, tree, network — and the epoch you
   stopped at. *"The CNN reached 0.94 against the tree's 0.82, stopped at epoch 5 with both curves still
   falling"* is a finding; *"we trained a CNN"* is not.
2. **Say where you got stuck and what you had already ruled out.** An unfinished step costs nothing;
   an unfinished step you cannot describe costs the write-up mark.

**What we found:** *(one sentence — baseline, tree, network, and the epoch)*

**What we do not trust:** *(one limitation — the epochs, the split, the metric, or a curve you did not like)*

**Where we got stuck:** *(if a step defeated you, say which and why — this is worth writing down)*

---
## 🟠 After the hour — presenting, and handing in

**Everything about how this is presented, questioned and marked lives in one place, and it is not this
notebook:**

📄 **[How the assignment works, and what to hand in](https://classes.incortx.com/DataAnalytics/session-00/)**

That page is the only version — it covers the minutes on screen, the code questions, asking
questions while other groups present, and every requirement for the hand-in. **Read it once at the
start of term, and again before your first turn.**

The three dates you need, and nothing else:

| When | What |
|---|---|
| **Before you leave today** | this notebook, as a file — *File → Download → Download .ipynb* |
| **Two days before session 5** | the homework: notebook, slides, video, README — **as files, no links** |
| **The start of session 5** | you present, from the notebook you handed in |

---
# Session 4 Wrap-Up

### Four things to remember

1. **A loss that fell is not a model that works.** A.1's loss reached 0.17 and the network lost to the
   tree; only the validation line said why.
2. **Read the curve before the number.** Where the validation loss turns is where training should stop —
   and both lines still falling means the number you have is not the number you would get.
3. **Flatten throws away where things are.** Two pixels cost a dense layer 30 points; a convolution
   slides, so it finds the edge wherever the edge is.
4. **The model you already know goes beside the new one.** On 891 rows the tree won; on 12,000 pictures
   the network won by 11 points. Neither is the general rule — the table is.

### What this session's notebook should end up carrying
- pixels in 0–1 or columns scaled, **and a validation split**, with a line saying why
- **`baseline()` and a tree** under the network's number
- **a learning curve with the epoch you chose marked on it**

*(Dates, file formats and everything else about handing in: see the section above.)*

### Next session — making things that never existed
Every model so far told things apart. **Next time the network builds** — an autoencoder that squeezes a
picture down to a few numbers and draws it back, a GAN that draws pictures nobody took, and then rows
that come in order, where the model has to remember.